# Fraud model — Phase 2 temporal training

**Status: prepared for execution after a successful Serverless environment v4 smoke test.** This notebook trains the required baselines, regularized logistic-regression candidates, and decision-tree candidates on the fixed temporal split. Preprocessing is fitted on train only. Hyperparameters and model choice use validation only. The final-test period is excluded from the source query and must remain untouched until Phase 3.

## Questions this experiment answers

1. Does either required model improve over constant/always-negative baselines on temporal validation?
2. Which candidate gives the best validation ranking quality, and what does the stated 10% review-capacity assumption imply?
3. Do results vary materially by country or channel, with denominators shown?

## Method and safeguards

- Run on Databricks Serverless environment version 4, which supports PySpark ML and MLflow for Spark.
- Read one pinned Delta snapshot (version 1); use a single transactions table, so joins cannot multiply rows.
- Train: before 2025-07-01; validation: 2025-07-01 through 2025-12-31; no final-test data.
- Fit median imputers and categorical vocabularies on train only; one-hot encode categories and preserve unknowns.
- Compare B0/B1, six M1 configurations, and twelve M2 configurations; balanced class weights affect training only. No sampling.
- Primary metric is Spark `areaUnderPR` (stepwise precision-recall area); ROC-AUC is reported for the selected candidate. Results are offline synthetic-data evidence, not calibrated probabilities or a production fraud decision.
- Log attempt parameters, outcomes, metrics, selected Spark pipeline, and bounded aggregate evidence in MLflow under `/Shared/fraud-eda/phase2-training`.
- The model artifact is not registered, and no database table is written.


In [ ]:
%run ./train_phase2

In [ ]:
import json
import mlflow
result = run_training(spark, mlflow)
print(json.dumps(result, indent=2, allow_nan=False))
dbutils.notebook.exit(json.dumps(result, allow_nan=False))